In [11]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from collections import Counter
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight

In [12]:
df = pd.read_csv('../data/红酒品质分类.csv')
# df.info()

x = df.iloc[:, :-1]
y = df.iloc[:, -1] - 3

# print(x[:5])
# print(y[:5])
# print(Counter(y))

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=23, stratify=y)
pd.concat([x_train, y_train], axis=1).to_csv('../data/红酒品质分类_train.csv', index=False)
pd.concat([x_test, y_test], axis=1).to_csv('../data/红酒品质分类_test.csv', index=False)

In [13]:
train_data = pd.read_csv('../data/红酒品质分类_train.csv')
test_data = pd.read_csv('../data/红酒品质分类_test.csv')

x_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]

x_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]

estimator = xgb.XGBClassifier(
    max_depth=5, n_estimators=100,
    learning_rate=0.1, random_state=23,
    objective='multi:softmax',
)

sample_weight = class_weight.compute_sample_weight('balanced', y_train)

estimator.fit(x_train, y_train, sample_weight=sample_weight)
print(f'Acc: {estimator.score(x_test, y_test)}')
joblib.dump(estimator, '../data/红酒品质分类model.pkl')
print('model saved!')

Acc: 0.65625
model saved!


In [14]:
train_data = pd.read_csv('../data/红酒品质分类_train.csv')
test_data = pd.read_csv('../data/红酒品质分类_test.csv')

x_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]

x_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]

estimator = joblib.load('../data/红酒品质分类model.pkl')
param_dict = {'max_depth': [2, 3, 5, 7], 'n_estimators': [30, 50, 100, 150], 'learning_rate': [0.2, 0.3, 1, 1.3]}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)
gs_estimator = GridSearchCV(estimator, param_dict, cv=skf)

gs_estimator.fit(x_train, y_train)
print(f'Acc: {gs_estimator.score(x_test, y_test)}')


Acc: 0.684375
